# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import sys
sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
import utilities
from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema

## Part 1: Bloom Detection Method Testing

### Threshold Method

#### Function for determining the climatological threshold value
Based on the climatological median, this function finds a certain percentage of that median and adds it to the median to get a threshold value for determining phytoplankton blooms
* Must include:
    * Threshold percentage = thld (thld=0.05 by default)
    * Path to climatology files = path (set to NES Annual Climatology 1997 - 2020 by default)

In [ ]:
#Define a function to return the threshold value based on the regional climatology.
def threshold_value(thld = 0.05, path = None):
    if path == None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return(thld_value)

#### Plotting the Climatological Threshold

In [ ]:
clim_med = threshold_value().squeeze()
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
bathym=unary_union(list(bathym.geometries()))
fig = plt.figure(figsize=(15,8))
map_projection = cartopy.crs.PlateCarree()
ax = plt.axes(projection=map_projection)
im = plt.pcolormesh(clim_med.lon,
                    clim_med.lat,
                    clim_med,
                    cmap=cmocean.cm.algae,
                    norm=LogNorm(vmin=0.1, vmax=10.0)
)
custom_ticks = [0.1,1,10]
cb = plt.colorbar(im,shrink=0.8,label='Chlorophyll a Threshold ($mg/m^3$)',ticks=custom_ticks,format='%g') #$ $ makes it a LaTEX function so it actually formats as an equation

ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree()) #Adding the shelf break line
ax.set_extent([-77,-62,37,47])
ax.gridlines(draw_labels=True)
ax.set_title('Chlorophyll a Climatological Median Threshold', fontsize=24)

#### Create a mask to filter data for bloom conditions

This function (bloom_mask_Boolean) takes the input data and determines if values exceed the climatological threshold determined above (or found via the threshold_value function). If the value is less than or equal to the threshold, then it is reported as false. If it is greater than the threshold, it is reported as true. You must run the threshold function prior to running this function as it requires its value to be saved as clim_med.
* Must include
    * Path to files = path (set to D8 NES shelf files by default)

In [ ]:
def bloom_mask_Boolean(path=None): #Produces True and False values
    if path == None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NWA
        ds = xr.open_mfdataset(files)
    else:
        ds = xr.open_mfdataset(path)
    med_CHL = ds.CHL_median #Extracts CHL_mean variable for the files
    is_bloom_CHL = med_CHL > clim_med #Is the median chl-a in each files greater than the climatological mean? Creates a Boolean array of trues and falses.
    return is_bloom_CHL

This function (bloom_mask_numeric) functions similar to bloom_mask_Boolean, except it reports false values as 0 and true values retain their actual value. You must run the threshold function prior to running this function as it requires its value to be saved as clim_med.
* Must include:
    * Path to files = path (set to D8 NES shelf files by default)

In [ ]:
def bloom_mask_numeric(path=None): #Produces 0 and actual values
    if path == None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NWA
        ds = xr.open_mfdataset(files)
    else:
        #ds = xr.open_mfdataset(path)
        ds = xr.open_zarr(path,consolidated=True) #Opens zarr file, use until mfdatasets is working properly
    med_CHL = ds.CHL_median #Extracts CHL_mean variable for the files
    clim_med_new = clim_med.squeeze('time', drop=True) #Removes time dimension from climatological mean, FIX THIS LINE
    is_bloom_CHL = med_CHL.where(med_CHL > clim_med_new, 0) #Turns false into 0 and trues retain their value
    return is_bloom_CHL

#### Mask Intervals
Finding the climatological threshold at a few different inteverals (5%, 10%, 15%, 20%, 25%, and 30%)

In [ ]:
thld_value = [0.05,0.1,0.15,0.2,0.25,0.3]
clim_med = threshold_value()
bloom_5 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[1])
bloom_10 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[2])
bloom_15 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[3])
bloom_20 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[4])
bloom_25 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[5])
bloom_30 = bloom_mask_numeric(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')

Plotting all of the masks (5% - 30%)
<br> Must pick a day of the year from (INPUT VALUE RANGE HERE)

In [ ]:
DOY = 7181
datasets = [
    (bloom_5[DOY],"Chlorophyll a 5% Mask"),
    (bloom_10[DOY], "Chlorophyll a 10% Mask"),
    (bloom_15[DOY], "Chlorophyll a 15% Mask"),
    (bloom_20[DOY], "Chlorphyll a 20% Mask"),
    (bloom_25[DOY], "Chlorophyll a 25% Mask"),
    (bloom_30[DOY], "Chlorophyll a 30% Mask"),
    ]

fig, axes = plt.subplots(3,2,figsize=(14,12),subplot_kw={"projection":map_projection})
axes_flat = axes.flatten()

im= None

for i, (data,title) in enumerate(datasets):
    ax = axes_flat[i]
    im = ax.pcolormesh(data.lon,
                data.lat,
                data,
                cmap=cmocean.cm.algae,
                norm=LogNorm(vmin=0.1, vmax=10.0)
    )
    ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
    ax.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
    ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree())
    ax.set_xlabel('Longitude ($^o$)', fontsize=12)
    ax.set_ylabel('Latitude ($^o$)', fontsize=12)
    ax.set_extent([-77,-63,34.5,46])
    ax.set_title(title, fontsize=14) #Plot headings
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False

custom_ticks = [0.1,1,10]
cb = fig.colorbar(im,ax=axes,shrink=0.5,label='Chlorophyll a Concentration ($mg/m^3$)',ticks=custom_ticks,format='%g')
fig.suptitle("Chlorophyll a Masks Based on NES Annual Climatology",fontsize=20) #Overall figure heading

#### Histograms of Data

Create the functions to subset the data

In [14]:
daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)

Function to clip the median chl-a data for a specific longitude and latitude
* Must input 
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

In [ ]:
def hist_local_chl(lat_min,lat_max,lon_min,lon_max,path=None,region_title=None):
    if path == None:
        #daily_data = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        daily_data = xr.open_mfdataset(path)
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    daily_data_local = daily_data.CHL_median.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    daily_data_local = daily_data_local.mean(dim=['lat','lon'])
    return daily_data_local

Function to clip the climatological data to the area 
* Must include
    * Path to file: path =
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max


In [ ]:
def hist_clim_local(lat_min,lat_max,lon_min,lon_max,region_title=None,path=None):
    if path == None:
        clim = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
    else:
        clim = xr.open_dataset(path)
    clim_med = clim.CHL_median
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    clim_med = clim_med.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    clim_med_bounded = clim_med.mean(dim=['lat','lon'])
    return clim_med_bounded


Function builds a square (polygon) of the area to plot
* Must include
    * Minimum latitude: lat_min
    * Maximum latitude: lat_max
    * Minimum longitude: lon_min
    * Maximum longitude: lon_max

In [ ]:
def bound_local(lat_min,lat_max,lon_min,lon_max): #Builds polygon shape for the map
    coords = [(lon_min,lat_min),
              (lon_max,lat_min),
              (lon_max,lat_max),
              (lon_min,lat_max),
              (lon_min,lat_min)]
    box_polygon = Polygon(coords)
    return box_polygon

Assign variables to the various datasets for plotting

In [ ]:
daily_local_GOM = hist_local_chl(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68,region_title='Gulf of Maine')
daily_local_GB = hist_local_chl(lat_min=41.7,lat_max=40.7,lon_min=-68,lon_max=-67,region_title='Georges Bank')
daily_local_MAB = hist_local_chl(lat_min=39.5,lat_max=38.5,lon_min=-74,lon_max=-73,region_title='Middle Atlantic Bight')
clim_local_GOM = hist_clim_local(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68,region_title='Gulf of Maine')
clim_local_GB = hist_clim_local(lat_min=41.7,lat_max=40.7,lon_min=-68,lon_max=-67,region_title='Georges Bank')
clim_local_MAB = hist_clim_local(lat_min=39.5,lat_max=38.5,lon_min=-74,lon_max=-73,region_title='Middle Atlantic Bight')
location_GOM = bound_local(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68)
location_GB = bound_local(lat_min=41.7,lat_max=40.7,lon_min=-68,lon_max=-67)
location_MAB = bound_local(lat_min=39.5,lat_max=38.5,lon_min=-74,lon_max=-73)

In [ ]:
local_chl = [daily_local_GOM,daily_local_GB,daily_local_MAB]
clim_med = [clim_local_GOM,clim_local_GB,clim_local_MAB]
clim_5 = [clim_local_GOM*1.05,clim_local_GB*1.05,clim_local_MAB*1.05]
clim_10 = [clim_local_GOM*1.1,clim_local_GB*1.1,clim_local_MAB*1.1]
clim_15 = [clim_local_GOM*1.15,clim_local_GB*1.15,clim_local_MAB*1.15]
clim_20 = [clim_local_GOM*1.2,clim_local_GB*1.2,clim_local_MAB*1.2]
clim_25 = [clim_local_GOM*1.25,clim_local_GB*1.25,clim_local_MAB*1.25]
clim_30 = [clim_local_GOM*1.3,clim_local_GB*1.3,clim_local_MAB*1.3]

In [ ]:
#Set up figure
fig=plt.figure(figsize=(12,14))
ax1 = plt.subplot(2,2,1,projection=crs.PlateCarree())
ax2 = plt.subplot(2,2,2)
ax3 = plt.subplot(2,2,3)
ax4 = plt.subplot(2,2,4)
# Location Map
ax1.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax1.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax1.add_geometries(bathym, facecolor='none', edgecolor='grey', crs=cartopy.crs.PlateCarree())
ax1.add_geometries(location_GOM, facecolor='none',edgecolor='purple',crs=cartopy.crs.PlateCarree(), label='Gulf of Maine')
ax1.add_geometries(location_GB, facecolor='none',edgecolor='purple',crs=cartopy.crs.PlateCarree(), label='Georges Bank')
ax1.add_geometries(location_MAB, facecolor='none',edgecolor='purple',crs=cartopy.crs.PlateCarree(), label='Middle Atlantic Bight')
ax1.set_extent([-77,-62,37,47])
gl=ax1.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False
ax1.set_title('Chlorophyll a Climatological Median and Histogram Locations', fontsize=14)

#Histograms
hist_axes = [ax2,ax3,ax4]
region_title = ["Gulf of Maine","Georges Bank",'Middle Atlantic Bight']
for i, data in enumerate(local_chl):
    ax = hist_axes[i]
    ax.hist(data,bins=100)
    ax.set_xlim(0,4)
    ax.axvline(clim_med[i][0], color='red',label='Median') #Pulls first value in list and then the 1 value in that value
    ax.axvline(clim_5[i][0], color='purple', label='5% Threshold')
    ax.axvline(clim_10[i][0], color='yellow',label='10% Threshold')
    ax.axvline(clim_15[i][0], color='orange',label='15% Threshold')
    ax.axvline(clim_20[i][0], color='green',label='20% Threshold')
    ax.axvline(clim_25[i][0], color='pink',label='25% Threshold')
    ax.axvline(clim_30[i][0], color='turquoise',label='30% Threshold')
    ax.legend()
    ax.set_title(region_title[i])
plt.tight_layout()
fig.suptitle("Chlorophyll a Concentrations",fontsize=20)

#### Determining the percentage of datapoints that lie above each threshold
Using the variables defined above for plotting, compare each value in the data set to that area's climatological median threshold.
* Threshold options:
    * clim_med = climatological median of the area (must use index of 0-2 in order of GOM, GB, MAB)
    * clim_5 = 5% above the climatological median (Must use indexing for all threshold values)
    * clim_10 = 10% above climatological median
    * clim_15 = 15% above climatological median
    * clim_20 = 20% above climatological median
    * clim_25 = 25% above climatological median
    * clim_30 = 30% above climatological median
* Index options: 0, 1, 2
* Dataset options:
    * daily_local_GOM = Gulf of Maine subset data
    * daily_local_GB = Georges Bank subset data
    * daily_local_MAB = Middle Atlantic Bight subset data


In [ ]:
def percent_above_thld(threshold,index,data):
    threshold=threshold[index].values
    total_above = int((data>threshold).sum())
    percent = (total_above/len(data))*100
    return percent

Testing 5%, 10%, and 15% for each region

In [ ]:
GOM_5 = percent_above_thld(clim_5,0,daily_local_GOM)
print("Gulf of Maine 5%: " + str(GOM_5))
GOM_10 = percent_above_thld(clim_10,0,daily_local_GOM)
print("Gulf of Maine 10%: " + str(GOM_10))
GOM_15 = percent_above_thld(clim_15,0,daily_local_GOM)
print("Gulf of Maine 15%: " + str(GOM_15))
GB_5 = percent_above_thld(clim_5,1,daily_local_GB)
print("Georges Bank 5%: " + str(GB_5))
GB_10 = percent_above_thld(clim_10,1,daily_local_GB)
print("Georges Bank 10%: " + str(GB_10))
GB_15 = percent_above_thld(clim_15,1,daily_local_GB)
print("Georges Bank 15%: " + str(GB_15))
MAB_5 = percent_above_thld(clim_5,2,daily_local_MAB)
print("Middle Atlantic Bight 5%: " + str(MAB_5))
MAB_10 = percent_above_thld(clim_10,2,daily_local_MAB)
print("Middle Atlantic Bight 10%: " + str(MAB_10))
MAB_15 = percent_above_thld(clim_15,2,daily_local_MAB)
print("Middle Atlantic Bight 15%: " + str(MAB_15))

#### Shapefile Analysis

Creates the shapefile for each region

In [15]:
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

Clip the climatology data to the shapefile

In [24]:
clim_regional = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
clim_regional.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
clim_regional.rio.write_crs("epsg:4326", inplace=True)
clipped_MAB_s = clim_regional.rio.clip(MAB_south_loc.geometry, shapefile.crs, drop=True)
clim_MAB_s = clipped_MAB_s.CHL_median.mean(dim=['lat','lon'])
clipped_MAB_n = clim_regional.rio.clip(MAB_north_loc.geometry, shapefile.crs, drop=True)
clim_MAB_n = clipped_MAB_n.CHL_median.mean(dim=['lat','lon'])
clipped_GB = clim_regional.rio.clip(GB_whole_loc.geometry, shapefile.crs, drop=True)
clim_GB = clipped_GB.CHL_median.mean(dim=['lat','lon'])
clipped_GOM_w = clim_regional.rio.clip(GOM_west_loc.geometry, shapefile.crs, drop=True)
clim_GOM_w = clipped_GOM_w.CHL_median.mean(dim=['lat','lon'])
clipped_GOM_e = clim_regional.rio.clip(GOM_east_loc.geometry, shapefile.crs, drop=True)
clim_GOM_e = clipped_GOM_e.CHL_median.mean(dim=['lat','lon'])
clim_NES = clim_regional.rio.clip(NES.geometry, shapefile.crs, drop=True)
clim_NES = clim_NES.CHL_median.mean(dim=['lat','lon'])

Clip D8 data to the shapefiles

In [25]:
daily_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
daily_data.rio.write_crs("epsg:4326", inplace=True)
clipped_daily_MABS = daily_data.rio.clip(MAB_south_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_south = clipped_daily_MABS.CHL_median.mean(dim=['lat','lon'])
clipped_daily_MABN = daily_data.rio.clip(MAB_north_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_north = clipped_daily_MABN.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GB = daily_data.rio.clip(GB_whole_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GB_whole = clipped_daily_GB.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GOMW = daily_data.rio.clip(GOM_west_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_west = clipped_daily_GOMW.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GOME = daily_data.rio.clip(GOM_east_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_east = clipped_daily_GOME.CHL_median.mean(dim=['lat','lon'])
clipped_daily_NES = daily_data.rio.clip(NES.geometry.apply(mapping), shapefile.crs, drop=True)
NES_full = clipped_daily_NES.CHL_median.mean(dim=['lat','lon'])

In [27]:
local_chl = [MAB_south,MAB_north,GB_whole,GOM_west,GOM_east,NES_full]
clim_med = [clim_MAB_s,clim_MAB_n,clim_GB,clim_GOM_w,clim_GOM_e,clim_NES]
clim_5 = [clim_MAB_s*1.05,clim_MAB_n*1.05,clim_GB*1.05,clim_GOM_w*1.05,clim_GOM_e*1.05,clim_NES*1.05]
clim_10 = [clim_MAB_s*1.10,clim_MAB_n*1.10,clim_GB*1.10,clim_GOM_w*1.10,clim_GOM_e*1.10,clim_NES*1.10]
clim_15 = [clim_MAB_s*1.15,clim_MAB_n*1.15,clim_GB*1.15,clim_GOM_w*1.15,clim_GOM_e*1.15,clim_NES*1.15]
clim_20 = [clim_MAB_s*1.2,clim_MAB_n*1.2,clim_GB*1.2,clim_GOM_w*1.2,clim_GOM_e*1.2,clim_NES*1.2]
clim_25 = [clim_MAB_s*1.25,clim_MAB_n*1.25,clim_GB*1.25,clim_GOM_w*1.25,clim_GOM_e*1.25,clim_NES*1.25]
clim_30 = [clim_MAB_s*1.3,clim_MAB_n*1.3,clim_GB*1.3,clim_GOM_w*1.3,clim_GOM_e*1.3,clim_NES*1.3]

In [ ]:
fig=plt.figure(figsize=(12,14))
ax1 = plt.subplot(4,2,1,projection=crs.PlateCarree()) #Locations map
ax2 = plt.subplot(4,2,2) # MAB South
ax3 = plt.subplot(4,2,3) # MAB North
ax4 = plt.subplot(4,2,4) # GB
ax5 = plt.subplot(4,2,5) # GOM West
ax6 = plt.subplot(4,2,6) # GOM East
ax7 = plt.subplot(4,2,7) # Full NES
# Location Map
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
bathym=unary_union(list(bathym.geometries()))
map_projection = cartopy.crs.PlateCarree()
ax1.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax1.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax1.add_geometries(bathym, facecolor='none', edgecolor='grey', crs=cartopy.crs.PlateCarree())
MAB_south_loc.boundary.plot(ax=ax1, color='green', linewidth=1.5, label="Middle Atlantic Bight South")
MAB_north_loc.boundary.plot(ax=ax1, color='blue', linewidth=1.5, label="Middle Atlantic Bight North")
GB_whole_loc.boundary.plot(ax=ax1, color='orange', linewidth=1.5, label="Georges Bank")
GOM_west_loc.boundary.plot(ax=ax1, color='red', linewidth=1.5, label="Gulf of Maine West")
GOM_east_loc.boundary.plot(ax=ax1, color='magenta', linewidth=1.5, label="Gulf of Maine East")
ax1.set_extent([-77,-62,37,47])
ax1.legend(fontsize=8,bbox_to_anchor=(1,1),loc='upper left')
gl=ax1.gridlines(draw_labels=True)
gl.top_labels = False
gl.right_labels = False
ax1.set_title('Chlorophyll a Histogram Locations', fontsize=14)

#Histograms
hist_axes = [ax2,ax3,ax4,ax5,ax6,ax7]
region_title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank",'Gulf of Maine West',"Gulf of Maine East","NE Continental Shelf"]
for i, dataa in enumerate(local_chl):
    ax = hist_axes[i]
    ax.hist(dataa,bins=100)
    ax.set_xlim(0,4)
    ax.axvline(clim_med[i][0], color='red',label='Median') #Pulls first value in list and then the 1 value in that value
    ax.axvline(clim_5[i][0], color='purple', label='5% Threshold')
    ax.axvline(clim_10[i][0], color='yellow',label='10% Threshold')
    ax.axvline(clim_15[i][0], color='orange',label='15% Threshold')
    ax.axvline(clim_20[i][0], color='magenta',label='20% Threshold')
    ax.axvline(clim_25[i][0], color='pink',label='25% Threshold')
    ax.axvline(clim_30[i][0], color='turquoise',label='30% Threshold')
    ax.legend(fontsize=8)
    ax.set_title(region_title[i])
plt.tight_layout()
fig.suptitle("Chlorophyll a Concentrations",fontsize=20,y=1.05)

#### Calculating the percentage of points above the threshold
Using the variables defined above for plotting and percent_above_thld(), compare each value in the data set to that area's climatological median threshold.
* Threshold options: (Must use index to get the correct area)
    * clim_med = climatological median of the area
    * clim_5 = 5% above the climatological median
    * clim_10 = 10% above climatological median
    * clim_15 = 15% above climatological median
    * clim_20 = 20% above climatological median
    * clim_25 = 25% above climatological median
    * clim_30 = 30% above climatological median
* Index options: 0, 1, 2, 3, 4, 5 (in the order of: Middle Atlantic Bight South, Middle Atlantic Bight North, Georges Bank, Gulf of Maine West, Gulf of Maine East, NES)
* Dataset options:
    * MAB_south = Middle Atlantic Bight South data
    * MAB_north = Middle Atlantic Bight North data
    * GB_whole = Georges Bank data
    * GOM_west = Gulf of Maine West data
    * GOM_east = Gulf of Maine East data
    * NES_full = Full NES region data

In [ ]:
MABS_5 = percent_above_thld(clim_5,0,MAB_south)
print("Middle Atlantic Bight South 5%: " + str(MABS_5))
MABS_10 = percent_above_thld(clim_10,0,MAB_south)
print("Middle Atlantic Bight South 10%: " + str(MABS_10))
MABS_15 = percent_above_thld(clim_15,0,MAB_south)
print("Middle Atlantic Bight South 15%: " + str(MABS_15))
MABN_5 = percent_above_thld(clim_5,1,MAB_north)
print("Middle Atlantic Bight North 5%: " + str(MABN_5))
MABN_10 = percent_above_thld(clim_10,1,MAB_north)
print("Middle Atlantic Bight North 10%: " + str(MABN_10))
MABN_15 = percent_above_thld(clim_15,1,MAB_north)
print("Middle Atlantic Bight North 15%: " + str(MABN_15))
GBW_5 = percent_above_thld(clim_5,2,GB_whole)
print("Georges Bank 5%: " + str(GBW_5))
GBW_10 = percent_above_thld(clim_10,2,GB_whole)
print("Georges Bank 10%: " + str(GBW_10))
GBW_15 = percent_above_thld(clim_15,2,GB_whole)
print("Georges Bank 15%: " + str(GBW_15))
GOMW_5 = percent_above_thld(clim_5,3,GOM_west)
print("Gulf of Maine West 5%: " + str(GOMW_5))
GOMW_10 = percent_above_thld(clim_10,3,GOM_west)
print("Gulf of Maine West 10%: " + str(GOMW_10))
GOMW_15 = percent_above_thld(clim_15,3,GOM_west)
print("Gulf of Maine West 15%: " + str(GOMW_15))
GOME_5 = percent_above_thld(clim_5,4,GOM_east)
print("Gulf of Maine East 5%: " + str(GOME_5))
GOME_10 = percent_above_thld(clim_10,4,GOM_east)
print("Gulf of Maine East 10%: " + str(GOME_10))
GOME_15 = percent_above_thld(clim_15,4,GOM_east)
print("Gulf of Maine East 15%: " + str(GOME_15))
NES_5 = percent_above_thld(clim_5,5,NES_full)
print("NES 5%: " + str(NES_5))
NES_10 = percent_above_thld(clim_10,5,NES_full)
print("NES 5%: " + str(NES_10))
NES_15 = percent_above_thld(clim_15,5,NES_full)
print("NES 5%: " + str(NES_15))

#### Plotting the histograms centered on the median
This centers the data at the median, making that area's median value equal to 0. Since the percent_deviation function calculates the percentage, each bin represents the amount of days spent in each percentage threshold (5%, 10%, 15%, 20%, 25%, 30%).

In [ ]:
def percent_deviation(dataset,index,median):
    top = dataset.squeeze()-median[index][0]
    fraction = top/median[index][0]
    percent_dev = fraction*100
    return percent_dev

In [ ]:
MAB_south_per_dev = percent_deviation(MAB_south,0,clim_med)
MAB_north_per_dev = percent_deviation(MAB_north,1,clim_med)
GB_per_dev = percent_deviation(GB_whole,2,clim_med)
GOM_west_per_dev = percent_deviation(GOM_west,3,clim_med)
GOM_east_per_dev = percent_deviation(GOM_east,4,clim_med)
NES_per_dev = percent_deviation(NES_full,5,clim_med)

Plotting the histograms

In [ ]:
custom_bins=[0,5,10,15,20,25,30,35]
fig,axes=plt.subplots(2,3,figsize=(25,14))
axes=axes.flatten() #Creates a 1-D numpy array of indices for axes instead of a 2 by 3 array
axes[0].hist(MAB_south_per_dev, bins=custom_bins, edgecolor='black', linewidth=1)
axes[0].set_title("Middle Atlantic Bight South")
axes[1].hist(MAB_north_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[1].set_title("Middle Atlantic Bight North")
axes[2].hist(GB_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[2].set_title("Georges Bank")
axes[3].hist(GOM_west_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[3].set_title("Gulf of Maine West")
axes[4].hist(GOM_east_per_dev,bins=custom_bins, edgecolor='black', linewidth=1)
axes[4].set_title("Gulf of Maine East")
axes[5].hist(NES_per_dev,bins=custom_bins,edgecolor='black',linewidth=1)
axes[5].set_title("NES")
fig.suptitle("Percentage Threshold from Regional Median", fontsize=20)

### Rate of Change

#### Spatially averaged rate of change

The rate_of_change function spatially averages the chlorophyll a data, smooths the data using a lowess smoother, and then find the rates of change between each data point, creating an array of rates of change.
* Must include:
    * lat_min
    * lat_max
    * lon_min
    * lon_max
    * path (set to weekly climatology by default)

In [ ]:
def rate_of_change(lat_min,lat_max,lon_min,lon_max,path=None):
    if path == None:
        files = get_prod_files('CHL',map_region='NES',period='WEEK')
        clim_med = xr.open_mfdataset(files)
    else:
        clim_med = xr.open_mfdataset(path)
    clim_med = clim_med.CHL_median
    clim_med = hist_local_chl(lat_min,lat_max,lon_min,lon_max,path)
    time = clim_med.time.astype('int64') #Changes time values to integers for smoothing
    clim_median=clim_med.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(clim_median['Chl_a'],time,frac=0.03)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    clim_roc = np.gradient(smoothed_CHL) #Calculates rate of change
    return clim_roc

The max_roc function builds on the rate_of_change function and finds the maximum rates of change in the array based on the parameters set in the function.
* Must include:
    * lat_min
    * lat_max
    * lon_min
    * lon_max
    * path (set to weekly climatology by default)
    * Distance = number of days between peaks (set to 10 days by default)
    * Prominence = the compared value of a rate of change to the next in order for it to be a peak (set to 0.02 by default)

In [ ]:
def max_roc(lat_min,lat_max,lon_min,lon_max,path,days=10,prm=0.02): #Days between peaks and the relative height of each peak value
    roc = rate_of_change(lat_min,lat_max,lon_min,lon_max,path)
    roc_max = find_peaks(roc,distance=days,prominence=prm)
    return roc_max

Using Lowess smoothing function

In [ ]:
data = hist_local_chl(lat_min=44,lat_max=43,lon_min=-69,lon_max=-68,path=r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_2007*.nc') #Slices chl-a data into a 1 degree box (in Gulf of Maine)
plt.figure(figsize=(10,8))
plt.plot(data.time.values,data,label="Raw Chl-a") #Plots the time series of raw data
time=data.time.astype('int64') # Transforms datetime into integers for smoothing (takes the dates in data and makes them integer values)
smoothed = sm.nonparametric.smoothers_lowess.lowess(data,time,frac=0.04) #smooths data (y-axis, x-axis, fraction) #0.03 provides the best fit to the graph while still smoothing out some of the little bumps
x_smooth = smoothed[:,0] #Separates the data, grabbing the time values
y_smooth = smoothed[:,1] #Separates the data, grabbing the chlorophyll values
plt.plot(pd.to_datetime(x_smooth),y_smooth, label="Smoothed Chl-a")
plt.axhline(clim_GOM,c="purple",label="Climatological median")
plt.axhline(clim_5[0],c="red",label="5% Threshold")
plt.axhline(clim_10[0],c="yellow",label="10% Threshold")
plt.axhline(clim_15[0],c="pink",label="15% Threshold")
plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
plt.xlabel("Date")
plt.title("Chlorophyll a in the Gulf of Maine in 2015")
plt.legend(fontsize=6)

#### Regional Rates of Change

In [ ]:
time_series = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]

In [ ]:
time_series_years = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
three_year_centered_data = []
for year in time_series_years:
    year_before = year-1
    year_after = year+1
    zarr_files=[rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_{year_before}_combined.zarr',rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_{year}_combined.zarr',rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_{year_after}_combined.zarr']
    data_3year = xr.open_mfdataset(zarr_files,engine="zarr")
    three_year_centered_data.append(data_3year)

In [ ]:
time_series_values = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
shapefile_geometries = [MAB_south_loc, MAB_north_loc, GB_whole_loc, GOM_west_loc, GOM_east_loc]
region_title = ['MABS','MABN','GB','GOMW','GOME']
for year in time_series_values:
    year_index = time_series_values.index(year)
    dataset = three_year_centered_data[year_index]
    for i in range(5):
        region_name = region_title[i]
        shapefile_geometry = shapefile_geometries[i]
        dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
        dataset.rio.write_crs("epsg:4326", inplace=True)
        clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile.crs, drop=True)
        regional_data = clipped_daily.mean(dim=['lat','lon'])   
        time = regional_data.time.astype('int64')
        median=regional_data.to_dataframe().reset_index()
        median = median["CHL_median"]
        smoothed_CHL = sm.nonparametric.smoothers_lowess.lowess(median,time,frac=0.04,return_sorted=False)
        df_output = pd.DataFrame({
            'time':time,
            'CHL_median':smoothed_CHL
        })
        df_output.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\smoothed_3year_{str(region_name)}_data_{time_series_years[year_index]}.csv',index=False)
        print(f"Successfully saved smoothed_3year_{str(region_name)}_data_{time_series_values[year_index]}.csv")

In [ ]:
def instant_rate_of_change(dataset):
    time = dataset.time.astype('int64') #Changes time values to integers for smoothing
    median=dataset.to_dataframe(name='Chl_a') #Converts it into a pandas dataframe
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(median['Chl_a'],time,frac=0.04)
    smoothed_CHL = smoothed_median[:,1] #Extracts Chl-a data from smooth curve
    roc = np.gradient(smoothed_CHL) #Calculates rate of change
    return roc

In [ ]:
def max_roc_dates(dataset,prm=0.02,days=10,): #Days between peaks and the relative height of each peak value
    roc = instant_rate_of_change(dataset)
    roc_max = find_peaks(roc,distance=days,prominence=prm)
    return roc_max

This function finds start and end dates of blooms

In [16]:
def bounding_data(dataset,shapefile_geometry):
    dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    dataset.rio.write_crs("epsg:4326", inplace=True)
    clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile.crs, drop=True)
    regional_year = clipped_daily.CHL_median.mean(dim=['lat','lon'])
    return regional_year

In [ ]:
def smoothing_data(dataset,shapefile_geometry,frac=0.04):
    time = dataset.time.astype('int64') #Changes time values to integers for smoothing
    dataset = bounding_data(dataset,shapefile_geometry)
    median=dataset.to_dataframe() #Converts it into a pandas dataframe
    median=median['CHL_median']
    smoothed_median = sm.nonparametric.smoothers_lowess.lowess(median,time,frac=frac)
    return smoothed_median

In [ ]:
values= [0.000706,0.00087, 0.00106, 0.013,0.00141]
shapefile_geometries = [MAB_south_loc, MAB_north_loc, GB_whole_loc, GOM_west_loc, GOM_east_loc]
region_title = ['MABS','MABN','GB','GOMW','GOME']
for value in values:
    for x in range(5):
        daily_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
        daily_data.rio.write_crs("epsg:4326", inplace=True)
        clipped_daily = daily_data.rio.clip(shapefile_geometries[x].geometry.apply(mapping), shapefile.crs, drop=True)
        regional_data = clipped_daily.mean(dim=['lat','lon'])   
        time = regional_data.time.astype('int64')
        median=regional_data.to_dataframe().reset_index()
        median = median["CHL_median"]
        smoothed_CHL = sm.nonparametric.smoothers_lowess.lowess(median,time,frac=value,return_sorted=False)
        df_output = pd.DataFrame({
            'time':time,
            'CHL_median':smoothed_CHL
        })
        df_output.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\smoothed_full_{region_title[x]}_data_frac{value}.csv',index=False)
        print("Successfully saved")

In [133]:
def bloom_peak_detection(dataset,index,days=10,prm=0.02):
    smoothed_CHL = dataset['smoothed_sg_win_15_poly_3']
    chl_peak_loc, _ = find_peaks(smoothed_CHL,distance=days,prominence=prm) #Finds all peaks
    clim_threshold = clim_10[index].values
    chl_peaks = []
    for i in range(0,len(chl_peak_loc)): #Finds all peaks above the threshold and removes peaks below the threshold
        if smoothed_CHL[chl_peak_loc[i]]>clim_threshold[0]:
            chl_peak_location = chl_peak_loc[i]
            chl_peaks.append(chl_peak_location)
        else:
            continue
    return chl_peaks

In [134]:
def bloom_event_detection(dataset,index,event_distance=14,peak_window=10):
    chl_peaks = bloom_peak_detection(dataset,index)
    time = dataset['time']
    if not chl_peaks:
        return []
    smoothed_CHL = dataset['smoothed_sg_win_15_poly_3']
    clim_threshold = float(clim_10[index].values[0])
    bloom_events = []
    current_event = [chl_peaks[0]]
    days_between_events = event_distance
    for i in range(1,len(chl_peaks)):
        previous_peak = chl_peaks[i-1]
        current_peak = chl_peaks[i]
        chl_between_peaks = smoothed_CHL[previous_peak:current_peak]
        dropped_below_thld = False
        if len(chl_between_peaks)>=peak_window:
            for start_day in range(len(chl_between_peaks)-peak_window+1):
                window = chl_between_peaks[start_day:start_day+peak_window] #Creates a rolling 10 day window to check for consecutive days below the threshold
                if all(chl_val<clim_threshold for chl_val in window):
                    dropped_below_thld = True
                    break
        if current_peak-previous_peak<days_between_events or not dropped_below_thld:
            #current_peak = pd.Timestamp(time[current_peak]) 
            #current_peak= (current_peak - pd.Timestamp("1970-01-01")).days
            current_event.append(current_peak)
        else:
            #current_peak = pd.Timestamp(time[current_peak]) 
            #current_peak= (current_peak - pd.Timestamp("1970-01-01")).days
            bloom_events.append(current_event)
            current_event = [current_peak] #If peak is not close to other peaks, it adds it to the event list by itself
    bloom_events.append(current_event)
    return bloom_events


In [135]:
def year_bounded_peaks(dataset,index,year):
    bloom_event_1 = bloom_event_detection(dataset,index)
    smoothed_time = dataset['time']
    year_peaks = []
    for i in bloom_event_1:
        if len(i) == 1:
            event_day = smoothed_time[i].iloc[0]
            peak_dates = pd.to_datetime(event_day)
            if peak_dates.year == year:
                peak_date_DOY = pd.to_datetime(peak_dates).dayofyear
                year_peaks.append(int(peak_date_DOY))
            else:
                continue
        else:
            for j in i:
                event_day = smoothed_time[j]
                peak_dates = pd.to_datetime(event_day)
                if peak_dates.year == year:
                    peak_date_DOY = pd.to_datetime(event_day).dayofyear
                    year_peaks.append(int(peak_date_DOY))
                else:
                    continue
    return year_peaks

In [136]:
def rolling_peak_window(dataset,index):
    event_window = 90
    peak_window = 10
    chl_median = dataset['smoothed_sg_win_15_poly_3']
    time = dataset['time']
    roc = np.gradient(chl_median)
    bloom_events = bloom_event_detection(dataset,index)
    start_of_window = max(0,bloom_events[0][0] - event_window)
    start_DOY = []
    end_DOY = []
    window_for_peak = peak_window
    last_end_day=start_of_window
    last_start_day=start_of_window
    clim_threshold = float(clim_10[index].values[0])

    for event in bloom_events:
        start_day = start_of_window
        for i in range(event[0]-1,window_for_peak-1,-1):
            if all((i+w <len(roc)) and roc[i+w]<0 for w in range(window_for_peak)):
                if i>=last_end_day and last_end_day>=last_start_day:
                    start_day = i + window_for_peak - 1
                    start_DOY.append(start_day)
                    last_start_day = i
                    break     
        if start_day == start_of_window and last_end_day>0 and event[0]>last_end_day:
                start_day = last_end_day
                start_DOY.append(start_day)
                last_start_day = start_day
        if start_day>=0:
            end_day = len(roc)-1
            for i in range(event[-1]+1,len(roc)-window_for_peak,1):
                roc_condition = all(roc[i+w]>=0 for w in range(window_for_peak))
                threshold_condition = all(chl_median[i+w]<clim_threshold for w in range(window_for_peak))
                if roc_condition and threshold_condition:
                    end_day=i
                    end_DOY.append(end_day)
                    last_end_day=end_day
                    break
    return start_DOY,end_DOY,bloom_events

In [139]:
MABS = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_smoother_tests.zarr')
start_DOY,end_DOY,bloom_events = rolling_peak_window(dataset=MABS,index=0)
print("Start days:")
print(start_DOY)
print("End days")
print(end_DOY)
print("Bloom events")
print(bloom_events)

Start days:
[24, 376, 671, 735, 1006, 1142, 1404, 1476, 1826, 1954, 2085, 2145, 2211, 2518, 2863, 3282, 3653, 3937, 4276, 4381, 4659, 4727, 4998, 5476, 5716, 5843, 6143, 6482, 6631, 6916, 7192, 7260, 7583, 7991, 8137, 8361, 8708, 9056, 9412, 9850, 10154]
End days
[226, 226, 671, 735, 893, 1060, 1330, 1330, 1330, 1476, 1476, 1602, 1602, 1954, 2085, 2085, 2145, 2187, 2481, 2481, 2728, 3226, 3226, 3540, 3540, 3540, 3540, 3910, 3910, 3910, 3910, 4254, 4254, 4254, 4254, 4351, 4540, 4727, 4875, 5338, 5338, 5716, 5716, 5763, 5763, 6023, 6457, 6457, 6631, 6631, 6631, 6731, 7115, 7115, 7260, 7474, 7474, 7474, 7846, 8137, 8161, 8511, 8967, 8967, 8967, 9266, 9617, 9617, 9999, 9999, 10235]
Bloom events
[[np.int64(57), np.int64(70)], [np.int64(100), np.int64(120), np.int64(130), np.int64(155), np.int64(171), np.int64(189), np.int64(201)], [np.int64(427), np.int64(444), np.int64(466), np.int64(492), np.int64(512), np.int64(530), np.int64(545)], [np.int64(722)], [np.int64(767), np.int64(780), np.int6

In [142]:
#big_data = rolling_peak_window(dataset=GB,index=2)
MABS['time']=pd.to_datetime(MABS['time'],unit='ns')
time = MABS['time']
time = pd.to_datetime(time)
days_1970 = pd.to_datetime("1970-01-01")
time_since_1970 = (time-days_1970).days
start_dates = []
end_dates = []
peak_dates = []
base_date = (pd.Timestamp("1997-09-04")-pd.Timestamp("1970-01-01")).days
for i, DOY in enumerate(start_DOY):
    start_date = start_DOY[i]
    peak_current= (base_date + start_date)
    start_dates.append(peak_current)
for i, DOY in enumerate(end_DOY):
    end_date = end_DOY[i]
    peak_current = base_date + end_date
    end_dates.append(peak_current)
for i, DOY in enumerate(bloom_events):
    if len(DOY) == 1:
        event_date = bloom_events[i][0]
        peak_value = (base_date + event_date)
        peak_dates.append(peak_value)

In [ ]:
peak_event_index = 3
figure = plt.figure(figsize=(14,10))
x_left_lim = peak_dates[peak_event_index] - 90
x_right_lim = peak_dates[peak_event_index] + 90
raw = bounding_data(daily_data,MAB_south_loc)
GB_CHL_smoothed = MABS['smoothed_sg_win_15_poly_3']
event = bloom_events
start_DOY_time = start_DOY
start_DOY_val = GB_CHL_smoothed[start_DOY_time]
start_DOY_date = time[start_DOY_time]
end_DOY_time = end_DOY
end_DOY_val = GB_CHL_smoothed[end_DOY_time]
end_DOY_date = time[end_DOY_time]
bloom_peak_time = event[peak_event_index]
bloom_peak_val = GB_CHL_smoothed[bloom_peak_time]
bloom_peak_date = time[bloom_peak_time]

plt.plot(raw.time.values,raw,label="Raw Chl-a",c='royalblue')
plt.plot(MABS.time.values, GB_CHL_smoothed,label='smoothed data',c='darkorange')
plt.axhline(clim_med[0][0],c="darkorchid",label="Climatological median")
plt.axhline(clim_10[0][0],c="red",label="10% Threshold")
#plt.scatter(roc_max_date,roc_max,c='green',s=100,zorder=5,marker='^',label="Max rates of change")
plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.xlim(left=x_left_lim)
plt.xlim(right=x_right_lim)
plt.title("Chlorophyll a in the MAB South")
plt.scatter(start_DOY_date,start_DOY_val,c='magenta',s=100,zorder=5,marker='s',label="Bloom start")
plt.scatter(end_DOY_date,end_DOY_val,c='darkblue',s=125,zorder=5,marker='*',label="Bloom end")
plt.scatter(bloom_peak_date,bloom_peak_val,c='crimson',s=100,zorder=5,marker='D',label="Bloom peak")
plt.legend(fontsize=8)

#filename = f"GB_{str(i)}_frac04.png"
#plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\peak_graphs\{filename}')
#plt.close()

In [ ]:
#Bloom definition
def initiation_date(dataset,index,days=10,prm=0.02,peak_window=10,event_distance=14):
    data = dataset
    smoothed_data = smoothing_data(data)
    smoothed_CHL = smoothed_data[:,1]
    roc = instant_rate_of_change(data)
    bloom_events = bloom_event_detection(dataset,index)
    start_DOY = []
    end_DOY = []
    window_for_peak = peak_window
    last_end_day=0
    last_start_day=0
    clim_threshold = clim_10[index]

    for event in bloom_events:
        start_day = 0
        for i in range(event[0]-1,window_for_peak-1,-1):
            if all(roc[i+w]<0 for w in range(window_for_peak)):
                if i>=last_end_day and last_end_day>=last_start_day:
                    start_day = i
                    start_DOY.append(start_day)
                    last_start_day = i
                    break
        if start_day == 0 and last_end_day>0 and event[0]>last_end_day:
                start_day = last_end_day
                start_DOY.append(start_day)
                last_start_day = start_day
        if start_day>0:
            end_day = len(roc)-1
            for i in range(event[-1]+1,len(roc)-window_for_peak,1):
                roc_condition = all(roc[i+w]>=0 for w in range(window_for_peak)) # Ensures that the roc is becoming positive
                threshold_condition = all(smoothed_CHL[i+w]<clim_threshold for w in range(window_for_peak)) # Ensures termination date only occurs after a drop below the climatological median
                if roc_condition and threshold_condition:
                    end_day=i
                    end_DOY.append(end_day)
                    last_end_day=end_day
                    break
    return start_DOY,end_DOY,bloom_events


This function finds the rate of change for the smoothed chl-a data, then find the maximum rate of change only for peaks where the smoothed peak value reached above the 10% climatological threshold.

In [ ]:
def max_roc_for_bloom(dataset,index):
    start_day_bloom=rolling_peak_window(dataset,index=index)[0]
    end_day_bloom=rolling_peak_window(dataset,index=index)[1]
    chl_median = dataset['CHL_median']
    roc = np.gradient(chl_median)
    bloom_events = []
    for i in range(len(start_day_bloom)):
        start_day=start_day_bloom[i]
        if i <len(end_day_bloom):
            end_day=end_day_bloom[i]
        else:
            continue
        range_roc = roc[start_day:end_day]
        if len(range_roc)>0:
            range_max_roc = np.argmax(range_roc) #Finds local maximum rate of change for each detected bloom
            max_roc_index = start_day+range_max_roc #Gets the actual day of year value
            bloom_events.append(max_roc_index)
    return bloom_events

In [ ]:
smoothed_data=MABN
max_roc = max_roc_for_bloom(dataset=smoothed_data,index=2)
raw_data = bounding_data(dataset=daily_data,shapefile_geometry=GB_whole_loc)

smoothed = smoothed_data
smoothed_data['time']=pd.to_datetime(smoothed_data.iloc[:,0])
time_smoothed = smoothed['time']
chl_smoothed = smoothed['CHL_median']

roc_max_time = np.asarray(max_roc).astype(int)
roc_max_date = pd.to_datetime(time_smoothed[roc_max_time])
roc_max = chl_smoothed[roc_max_time]

start_DOY, end_DOY, bloom_peak = rolling_peak_window(smoothed_data,index=2)
start_DOY_time = np.asarray(start_DOY).flatten().astype(int)
start_DOY_date = pd.to_datetime(time_smoothed[start_DOY_time])
start_DOY_val = chl_smoothed[start_DOY_time]
end_DOY_time = np.asarray(end_DOY).flatten().astype(int)
end_DOY_date = pd.to_datetime(time_smoothed[end_DOY_time])
end_DOY_val = chl_smoothed[end_DOY_time]
bloom_peak_time = np.concatenate(bloom_peak if len(bloom_peak)>0 else [np.asarray([])]).astype(int)
bloom_peak_date = pd.to_datetime(time_smoothed[bloom_peak_time])
bloom_peak_val = chl_smoothed[bloom_peak_time]

fig=plt.figure(figsize=(18,10))
plt.plot(raw_data.time.values,raw_data,label="Raw Chl-a") #Plots the time series of raw data
plt.plot(pd.to_datetime(time_smoothed),chl_smoothed, label="Smoothed Chl-a")
plt.axhline(clim_med[2][0],c="purple",label="Climatological median")
plt.axhline(clim_10[2][0],c="red",label="10% Threshold")
plt.scatter(roc_max_date,roc_max,c='green',s=100,zorder=5,marker='^',label="Max rates of change")
plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.title("Chlorophyll a in the " + region_title + " in " + str(year))
plt.scatter(start_DOY_date,start_DOY_val,c='magenta',s=100,zorder=5,marker='s',label="Bloom start")
plt.scatter(end_DOY_date,end_DOY_val,c='darkblue',s=125,zorder=5,marker='*',label="Bloom end")
plt.scatter(bloom_peak_date,bloom_peak_val,c='crimson',s=100,zorder=5,marker='D',label="Bloom peak")
plt.legend(fontsize=8)

In [ ]:
time_series = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
#time_series =[1998,1999,2000,2001,2002]
title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank","Gulf of Maine West","Gulf of Maine East"]
file = ["MABS","MABN","GB","GOMW","GOME"]
shapefile_location = [MAB_south_loc,MAB_north_loc,GB_whole_loc,GOM_west_loc,GOM_east_loc]
for year in time_series:
    year_index = time_series.index(year)
    for x in range(5):
        region_title = title[x]
        smoothed_data=pd.read_csv(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\smoothed_3year_{file[x]}_data_{year}.csv')
        max_roc = max_roc_for_bloom(dataset=smoothed_data,index=x,year=year)
        raw_data = bounding_data(year_index=year_index,shapefile_geometry=shapefile_location[x])

        smoothed = smoothed_data
        smoothed_data['time']=pd.to_datetime(smoothed_data.iloc[:,0])
        time_smoothed = smoothed['time']
        chl_smoothed = smoothed['CHL_median']

        #Converting ROC day of year into date format
        base_date = np.datetime64(f'{year}-01-01')
        roc_max_time = np.asarray(max_roc).astype(int)
        roc_max_date = pd.to_datetime(time_smoothed[roc_max_time])
        roc_max = chl_smoothed[roc_max_time]

        #Bloom start and end date
        start_DOY, end_DOY, bloom_peak = rolling_peak_window(smoothed_data,index=x,year=year)
        start_DOY_time = np.asarray(start_DOY).flatten().astype(int)
        start_DOY_date = pd.to_datetime(time_smoothed[start_DOY_time])
        start_DOY_val = chl_smoothed[start_DOY_time]
        end_DOY_time = np.asarray(end_DOY).flatten().astype(int)
        end_DOY_date = pd.to_datetime(time_smoothed[end_DOY_time])
        end_DOY_val = chl_smoothed[end_DOY_time]
        bloom_peak_time = np.concatenate(bloom_peak if len(bloom_peak)>0 else [np.asarray([])]).astype(int)
        bloom_peak_date = pd.to_datetime(time_smoothed[bloom_peak_time])
        bloom_peak_val = chl_smoothed[bloom_peak_time]

        #Plotting
        fig=plt.figure(figsize=(18,10))
        plt.plot(raw_data.time.values,raw_data,label="Raw Chl-a") #Plots the time series of raw data
        plt.plot(pd.to_datetime(time_smoothed),chl_smoothed, label="Smoothed Chl-a")
        plt.axhline(clim_med[x][0],c="purple",label="Climatological median")
        plt.axhline(clim_10[x][0],c="red",label="10% Threshold")
        plt.scatter(roc_max_date,roc_max,c='green',s=100,zorder=5,marker='^',label="Max rates of change")
        plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
        plt.xlabel("Date")
        plt.xticks(rotation=45)
        plt.title("Chlorophyll a in the " + region_title + " in " + str(year))
        plt.scatter(start_DOY_date,start_DOY_val,c='magenta',s=100,zorder=5,marker='s',label="Bloom start")
        plt.scatter(end_DOY_date,end_DOY_val,c='darkblue',s=125,zorder=5,marker='*',label="Bloom end")
        plt.scatter(bloom_peak_date,bloom_peak_val,c='crimson',s=100,zorder=5,marker='D',label="Bloom peak")
        plt.legend(fontsize=8)

        #axes[1].plot(pd.to_datetime(time_smoothed),roc_smooth)
        #roc_max = roc[roc_max_time]
        #y=0
        #axes[1].scatter(roc_max_date,roc_max,c='green',s=25,zorder=5,label="Max rates of change")
        #axes[1].axhline(y,c='purple')
        #axes[1].set_title("Rate of Change for " + region_title + " in " + str(year))
        #axes[1].legend
        #axes[1].set_ylabel("Rate of Change per day")
        #axes[1].set_xlabel("Date")

        filename = f"{str(file[x])}_{str(year)}_frac04.png"
        plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Start_end_graphs\{filename}')
        plt.close()

## Part 2: Quantify the Number of Phytoplankton Bloom Days Per Year

#### Actual number of bloom days per year over the time series

Based on the method(s) chosen in Part 1:
1. For each year: 365 - bloom
1. Plot the time series of the data to verify qualitatively

In [ ]:
def bloom_days_per_year(dataset,index,year):
    bloom_start_day , bloom_end_day, bloom_events = rolling_peak_window(dataset,index,year=year)
    number_bloom_days = []
    days_in_year = pd.to_datetime(f"{year}-12-31").dayofyear
    for x in range(len(bloom_start_day)):
        if x < len(bloom_end_day):
            bloom_start_date = int(bloom_start_day[x])
            start_date = pd.to_datetime(dataset['time'].iloc[bloom_start_date])
            if start_date.year == year:
                start_day = str(start_date.dayofyear)
            elif start_date.year<year:
                start_day = 1
            else:
                continue
            bloom_end_date = int(bloom_end_day[x])
            end_date = pd.to_datetime(dataset['time'].iloc[bloom_end_date])
            if end_date.year == year:
                end_day = str(end_date.dayofyear)
            elif end_date.year>year:
                end_day = days_in_year
            else:
                end_day = start_day
            amount_bloom_days = int(end_day)-int(start_day)
            number_bloom_days.append(amount_bloom_days)
        else:
            start_day = bloom_start_day[x]
            end_day = days_in_year
            amount_bloom_days = int(end_day)-int(start_day)+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_year=sum(number_bloom_days)
    return bloom_days_per_year

In [ ]:
time_series_values = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
bloom_days_GOMW = []
for year in time_series_values:
    data = pd.read_csv(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\smoothed_3year_GOMW_data_{year}.csv')
    blooms = bloom_days_per_year(data,index=3,year=year)
    bloom_days_GOMW.append(blooms)
    print(f"GOMW Number of bloom days for {year}: " + str(blooms))

In [ ]:
plt.scatter(time_series_values,bloom_days_GB)

Quantifying bloom days
1. Separate the data out by year
    1. Run the mask for just yearly data?
1. How many daily files meet the bloom criteria?

Plotting the time series

#### Average number of bloom days over the time series

Using the data from above:
1. Calculate the mean (median) number of bloom days for the time series
1. Create an array of the actual days of year
1. Run median (and mean) statistics on the array

## Part 3: Quantify the Number of Phytoplankton Blooms Per Year

#### Actual number of phytoplankton blooms per year over the time series

1. Find the number of peaks (where derivative is 0) above the threshold/place where bloom conditions begin
2. Verify qualitatively with time series

#### Average number of phytoplankton blooms per year over the time series

## Part 4: Bloom Characteristics Analysis

#### Bloom start day

#### Duration of blooms

#### Maximum chlorophyll a

#### Minimum chlorophyll a

#### Mean chlorophyll a

#### Integrated chlorophyll a

#### Location of blooms
1. Find the center of gravity of major blooms

#### Periodicity of Blooms

In [71]:
MABS = bounding_data(daily_data,MAB_south_loc)
MABN = bounding_data(daily_data,MAB_north_loc)
GB = bounding_data(daily_data,GB_whole_loc)
GOMW = bounding_data(daily_data,GOM_west_loc)
GOME = bounding_data(daily_data,GOM_east_loc)

### LOWESS Smoother

In [ ]:
import itertools
from sklearn.metrics import mean_squared_error, r2_score
frac_values = [0.00097, 0.00107, 0.00117, 0.00127,0.00136, 0.00145] 
it_values = [3, 5]
region_data = [MABS,MABN,GB,GOMW,GOME]
region_titles = ['MABS','MABN','GB','GOMW','GOME']
for x in range(5):
    data = region_data[x]
    region_title = region_titles[x]
    time = data.time.astype('int64')
    median=data.to_dataframe().reset_index()
    median = median["CHL_median"].values

    # itertools.product creates every possible combination of your lists
    for frac, it in itertools.product(frac_values, it_values):
        print(f"Running LOWESS: frac={frac}, it={it}...")
        
        # 1. Create a boolean mask of valid (non-NaN) indices for both arrays
        valid_mask = ~np.isnan(median)
        
        #2. Slice both arrays to only include days with valid numbers
        clean_median = median[valid_mask]
        clean_time = time[valid_mask]

        # Run the LOWESS function
        # return_sorted=False ensures the output matches our input array exactly
        smoothed_output = sm.nonparametric.smoothers_lowess.lowess(
            endog=clean_median, 
            exog=clean_time, 
            frac=frac, 
            it=it, 
            return_sorted=False
        )

        smoothed_full = np.full(len(data['time']), np.nan)
        smoothed_full[valid_mask] = smoothed_output

        variable_name = f"smoothed_lowess_frac_{frac}_int_{it}"
        data[variable_name] = (['time'],smoothed_full)

    data.to_zarr(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\{region_title}_smoother_tests.zarr',mode='w')
    print(f"Successfully saved {region_title} zarr file with all testing variables")

### Savitsky-Golay Smoother

In [97]:
from scipy.signal import savgol_filter
region_data = [MABN,GB,GOMW,GOME]
region_titles = ['MABN','GB','GOMW','GOME']
for x in range(4):
    data = region_data[x]
    region_title = region_titles[x]
    time = data.time.astype('int64')
    median=data.to_dataframe().reset_index()
    median = median["CHL_median"].values
    mask = ~np.isnan(median)
    chl_interp = pd.Series(median).interpolate(method='linear').bfill().ffill().values
    window_lengths = [7,15]
    poly_order = [3]

    for window,poly in itertools.product(window_lengths,poly_order):
        sg_smoothed = savgol_filter(chl_interp,window_length=window,polyorder=poly)
        sg_smoothed_full = np.copy(sg_smoothed)
        sg_smoothed_full[~mask]=np.nan
        variable_name = f"smoothed_sg_win_{window}_poly_{poly}"
        data[variable_name]=(['time'],sg_smoothed_full)
    data.to_zarr(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\{region_title}_smoother_tests.zarr',mode='a')
    print("Success!")

c:\Users\grace.davis\AppData\Local\anaconda3\envs\satprocessing\Lib\site-packages\zarr\api\asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Success!


c:\Users\grace.davis\AppData\Local\anaconda3\envs\satprocessing\Lib\site-packages\zarr\api\asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Success!


c:\Users\grace.davis\AppData\Local\anaconda3\envs\satprocessing\Lib\site-packages\zarr\api\asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Success!


c:\Users\grace.davis\AppData\Local\anaconda3\envs\satprocessing\Lib\site-packages\zarr\api\asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Success!


In [112]:
time_series = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024]
region_titles = ["MAB South","MAB North","Georges Bank","Gulf of Maine West","Gulf of Maine East"]
region_acro = ["MABS","MABN","GB","GOMW","GOME"]
for x in range(5):
    region_title = region_titles[x]
    smoother = xr.open_zarr(rf'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\{region_acro[x]}_smoother_tests.zarr')
    for year in time_series:  
        start_year = year-1
        end_year = year+1
        fig, ax = plt.subplots(figsize=(16,8))
        smoother['CHL_median'].plot(ax=ax, c='darkorange',label="Raw data",alpha=0.5)
        smoother['smoothed_lowess_frac_0.00117_int_3'].plot(ax=ax, c='blue',label="Lowess Frac: 12 days, int: 3",alpha=0.8)
        #smoother['smoothed_lowess_frac_0.00145_int_3'].plot(ax=ax, c='green',label="Frac: 15 days, int: 3",alpha=0.8)
        smoother['smoothed_sg_win_7_poly_3'].plot(ax=ax, c='red',label="SG Window = 7 days",alpha=0.8)
        smoother['smoothed_sg_win_15_poly_3'].plot(ax=ax, c='green',label="SG Window = 15 days",alpha=0.8)
        plt.axhline(clim_med[x][0],c="purple",label="Climatological median",alpha=0.3)
        plt.axhline(clim_10[x][0],c="red",label="10% Threshold",alpha=0.3)
        start = np.datetime64(f'{str(start_year)}-01-01','D').astype(int)
        end = np.datetime64(f'{str(end_year)}-01-01','D').astype(int)
        plt.xlim(start,end)
        plt.xlabel("Date")
        plt.ylabel("CHL-a Concentrations ($mg/m^3$)")
        plt.legend(fontsize=8)
        plt.title(f"Smoothers for {region_title} from {start_year}-{end_year}")
        filename = f"{region_acro[x]}_smoothers_{str(start_year)}_{str(end_year)}"
        plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Smoother_testing\{filename}')
        plt.close()
